# Satellite Collision Risk Assessment and Prediction (SCRAP)
**CSAI 801 — Machine Learning | Queen's University, Winter 2026**  
Mahmoud Alyosify · Mohamed Yahya · Mirna Embaby

---

## Pipeline Overview

This notebook implements a two-stage supervised machine learning framework for predicting
the final collision risk between a satellite and space debris in Low Earth Orbit (LEO).

**Operational Constraint:** All predictions use only telemetry data available at least
two days prior to the Time of Closest Approach (TCA).

### Architecture

```
RAW CDMs (time-series per event)
        |
        v
[1] 2-Day Cutoff Filtration  -- drop CDMs inside TCA-2day window
        |
        v
[2] Physics Feature Engineering  -- Mahalanobis distance, covariance, orbital geometry
        |
        v
[3] Time-Series Flattening  -- 11 aggregation statistics per feature
        |
        v
[4] Stage 1: Sentinel  (LightGBM Classifier)
        |    Maximises F2 (recall-weighted) to minimise false negatives
        |
        +-- Predicted LOW RISK  --> clip to -6.001 (safe)
        |
        +-- Predicted HIGH RISK --> Stage 2
                                        |
                                        v
                                [5] Stage 2: Specialist  (XGBoost Regressor)
                                        Fine-grained risk magnitude prediction
                                        trained on high-risk events only
```

**Evaluation Metric (ESA Official):**
$$L = \frac{1}{F_2} \times \text{MSE}_{HR}$$
where $F_2$ uses $\beta = 2$ (recall weighted 4x over precision) and MSE is computed
in probability space over high-risk events only.


## Section 1 — Environment Setup

In [ ]:
# Install required packages
# Run this cell once per session if packages are not already installed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'xgboost', 'lightgbm', 'catboost', 'optuna',
                'shap', 'datasets', 'scipy', '-q'], check=False)
print('Environment ready.')


## Section 2 — Imports and Global Constants

In [ ]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import xgboost  as xgb
import lightgbm as lgb
import optuna
import shap

optuna.logging.set_verbosity(optuna.logging.WARNING)

from datasets              import load_dataset
from scipy.optimize        import differential_evolution
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.metrics       import (precision_score, recall_score,
                                   fbeta_score, confusion_matrix)

np.random.seed(42)
pd.set_option('display.max_columns', None)

# Matplotlib style — clean academic output
plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'font.size'        : 11,
})
PALETTE = {'hr': '#C0392B', 'lr': '#2980B9', 'neutral': '#2C3E50', 'accent': '#27AE60'}

# ── ESA Competition Constants ─────────────────────────────────────────────────
CUTOFF_DAYS    = 2.0          # operational 2-day forecast horizon
RISK_THRESHOLD = 1e-6         # ESA high-risk classification boundary
LOG_THRESHOLD  = -6.0         # log10(1e-6)
SIGMA_EPS      = 1e-10        # numerical stability floor
R_EARTH_KM     = 6378.137     # Earth's equatorial radius (km)
N_FOLDS        = 5            # StratifiedGroupKFold folds
BETA           = 2.0          # F_beta recall weight (ESA spec)
UNCERTAINTY_P  = 75           # percentile for high-uncertainty flag

# ── Hardware Detection ────────────────────────────────────────────────────────
def _detect_hardware():
    try:
        p = xgb.XGBRegressor(tree_method='hist', device='cuda', n_estimators=1)
        p.fit([[0]], [0])
        return 'cuda'
    except Exception:
        pass
    try:
        import torch
        if torch.backends.mps.is_available():
            return 'mps'
    except Exception:
        pass
    return 'cpu'

HW         = _detect_hardware()
XGB_DEVICE = 'cuda' if HW == 'cuda' else 'cpu'
LGB_DEVICE = 'gpu'  if HW == 'cuda' else 'cpu'
N_TRIALS   = 80 if HW == 'cuda' else 25

print(f'Hardware detected : {HW.upper()}')
print(f'XGBoost device    : {XGB_DEVICE}')
print(f'LightGBM device   : {LGB_DEVICE}')
print(f'Optuna trials     : {N_TRIALS}')
print(f'Risk threshold    : log10({RISK_THRESHOLD}) = {LOG_THRESHOLD}')


## Section 3 — Official Competition Loss Function

The ESA Kelvins competition evaluates submissions using:

$$L(\hat{r}) = \frac{1}{F_2} \times \text{MSE}_{HR}$$

**Key implementation details:**
- $\text{MSE}_{HR}$ is computed in **probability space** (i.e., on $10^y$, not on $y$ directly)
- The high-risk threshold is $r \geq 10^{-6}$, equivalent to $\log_{10}(r) \geq -6$
- $F_2$ uses $\beta = 2$, weighting recall four times more heavily than precision
- A composite score $S = F_2 / (1 + \text{MSE}_{HR}) \in [0, 1]$ is used internally
  for threshold and hyperparameter selection, since $L$ is numerically unstable when $F_2 \approx 0$


In [ ]:
def competition_loss(y_true_log10, y_pred_log10,
                     beta=BETA, threshold=LOG_THRESHOLD, verbose=True):
    """
    Official ESA competition loss: L = (1 / F2) * MSE_HR

    Parameters
    ----------
    y_true_log10 : array-like
        True risk values in log10 space.
    y_pred_log10 : array-like
        Predicted risk values in log10 space.
    beta : float
        F-beta parameter (default 2.0 per ESA spec).
    threshold : float
        Log10 classification boundary (default -6.0 = log10(1e-6)).
    verbose : bool
        Print detailed breakdown.

    Returns
    -------
    float : Loss L. Lower is better. Returns inf if F2 = 0 or no HR events.

    Notes
    -----
    MSE is computed in probability space (10^y), not log space.
    Predictions are clipped to [-50, 0] to prevent numerical overflow.
    """
    y_true = np.asarray(y_true_log10, dtype=float).ravel()
    y_pred = np.clip(np.asarray(y_pred_log10, dtype=float).ravel(), -50, 0)

    r_true = 10.0 ** y_true
    r_pred = 10.0 ** y_pred

    t_prob     = 10.0 ** threshold
    y_true_bin = (r_true >= t_prob).astype(int)
    y_pred_bin = (r_pred >= t_prob).astype(int)

    prec  = precision_score(y_true_bin, y_pred_bin, zero_division=0.0)
    rec   = recall_score   (y_true_bin, y_pred_bin, zero_division=0.0)
    denom = beta**2 * prec + rec
    f2    = 0.0 if denom == 0 else (1 + beta**2) * prec * rec / denom

    hr_mask = y_true_bin == 1
    n_hr    = int(hr_mask.sum())
    if n_hr == 0:
        if verbose:
            print('  Warning: No high-risk events in ground truth.')
        return float('inf')

    mse_hr = float(np.mean((r_true[hr_mask] - r_pred[hr_mask]) ** 2))
    loss   = float('inf') if f2 == 0.0 else (1.0 / f2) * mse_hr

    if verbose:
        tp = int(((y_true_bin == 1) & (y_pred_bin == 1)).sum())
        fp = int(((y_true_bin == 0) & (y_pred_bin == 1)).sum())
        fn = int(((y_true_bin == 1) & (y_pred_bin == 0)).sum())
        print(f'  Threshold        : log10(r) >= {threshold}  (r >= {10**threshold:.0e})')
        print(f'  High-risk events : {n_hr:>5} / {len(r_true)}')
        print(f'  TP={tp:>5}  FP={fp:>5}  FN={fn:>5}')
        print(f'  Precision        : {prec:.4f}')
        print(f'  Recall           : {rec:.4f}  [beta=2: recall weighted 4x over precision]')
        print(f'  F2 Score         : {f2:.4f}')
        print(f'  MSE_HR (prob)    : {mse_hr:.4e}  [probability space]')
        print(f'  Loss L           : {loss:.6f}  [lower is better]')
    return loss


def loss_components(y_true_log10, y_pred_log10,
                    beta=BETA, threshold=LOG_THRESHOLD):
    """
    Return (f2, mse_hr, L, S) without printing.
    S = F2 / (1 + MSE_HR) is the composite score used for
    threshold tuning and hyperparameter selection.
    """
    y_true = np.asarray(y_true_log10, dtype=float).ravel()
    y_pred = np.clip(np.asarray(y_pred_log10, dtype=float).ravel(), -50, 0)
    r_true = 10.0 ** y_true
    r_pred = 10.0 ** y_pred
    t_prob = 10.0 ** threshold
    y_true_bin = (r_true >= t_prob).astype(int)
    y_pred_bin = (r_pred >= t_prob).astype(int)
    prec  = precision_score(y_true_bin, y_pred_bin, zero_division=0.0)
    rec   = recall_score   (y_true_bin, y_pred_bin, zero_division=0.0)
    denom = beta**2 * prec + rec
    f2    = 0.0 if denom == 0 else (1 + beta**2) * prec * rec / denom
    hr_mask = y_true_bin == 1
    n_hr    = int(hr_mask.sum())
    if n_hr == 0:
        return 0.0, float('inf'), float('inf'), 0.0
    mse_hr = float(np.mean((r_true[hr_mask] - r_pred[hr_mask]) ** 2))
    L      = float('inf') if f2 == 0.0 else (1.0 / f2) * mse_hr
    S      = f2 / (1.0 + mse_hr)
    return f2, mse_hr, L, S


# Sanity check
_y = np.full(200, LOG_THRESHOLD - 1.0)
_y[:10] = LOG_THRESHOLD + 0.5
f2, mse, L, S = loss_components(_y, _y)
print(f'Sanity check (perfect prediction):')
print(f'  F2={f2:.4f}  MSE_HR={mse:.2e}  L={L:.6f}  S={S:.4f}')
print('Loss functions defined.')


## Section 4 — Data Loading

Dataset: ESA Historical Conjunction Data Messages (CDMs)  
Source: HuggingFace — `mahmoudalyosify/SCRAP`


In [ ]:
HF_DATASET_ID = 'mahmoudalyosify/SCRAP'

def load_split(split):
    """Load a train/test split from HuggingFace with Parquet fallback."""
    try:
        ds = load_dataset(HF_DATASET_ID, split=split)
        return ds.to_pandas()
    except Exception as e:
        print(f'  HuggingFace API failed: {e}')
        print('  Attempting Parquet fallback...')
        import urllib.request, io
        base = ('https://huggingface.co/datasets/'
                'mahmoudalyosify/SCRAP/resolve/main/data')
        buf = io.BytesIO()
        urllib.request.urlretrieve(
            f'{base}/{split}-00000-of-00001.parquet', buf)
        return pd.read_parquet(buf)

print('Loading training split...')
df_train_raw = load_split('train')
print(f'  Shape   : {df_train_raw.shape}')
print(f'  Events  : {df_train_raw["event_id"].nunique():,}')

print('Loading test split...')
df_test_raw = load_split('test')
print(f'  Shape   : {df_test_raw.shape}')
print(f'  Events  : {df_test_raw["event_id"].nunique():,}')
print('Data loading complete.')


## Section 5 — Preprocessing

- Numeric columns: median imputation for missing values
- `c_object_type`: ordinal encoding (DEBRIS=3, ROCKET BODY=2, PAYLOAD=1, UNKNOWN=0)


In [ ]:
C_OBJECT_MAP = {
    'UNKNOWN': 0, 'TBA': 0,
    'PAYLOAD': 1,
    'ROCKET BODY': 2,
    'DEBRIS': 3
}

def preprocess(df):
    df      = df.copy()
    num_cols = df.select_dtypes(include='number').columns.tolist()
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df['c_object_type'] = (
        df['c_object_type']
          .fillna('UNKNOWN').str.upper()
          .str.replace('UNKOWN', 'UNKNOWN', regex=False)
          .map(C_OBJECT_MAP).fillna(0).astype(int)
    )
    return df

df_train_raw = preprocess(df_train_raw)
df_test_raw  = preprocess(df_test_raw)

print(f'Preprocessing complete.')
print(f'  Numeric columns imputed with column median.')
print(f'  c_object_type mapped to ordinal: DEBRIS=3, ROCKET BODY=2, PAYLOAD=1, UNKNOWN=0')


## Section 6 — Physics-Informed Feature Engineering

Row-level features derived from orbital mechanics, computed before time-series aggregation.

| Feature | Physical Meaning |
|---|---|
| `mahalanobis_distance` | Covariance-normalised spatial separation |
| `uncertainty_volume` | log(sigma_R * sigma_T * sigma_N) — combined 3D error volume |
| `t/c_log_cov_det` | Log-determinant of position covariance matrix |
| `combined_sigma_rdot` | Combined radial velocity uncertainty |
| `t/c_h_apo`, `t/c_h_per` | Apogee and perigee altitudes above Earth surface |
| `inc_difference` | Inclination delta between target and chaser |


In [ ]:
def add_physics_features(df):
    """
    Compute physics-informed row-level features.
    All features are derived from quantities available in a standard CDM.
    """
    df = df.copy()

    # Combined RTN position uncertainty
    for ax in ['r', 't', 'n']:
        df[f'combined_sigma_{ax}'] = np.sqrt(
            df[f't_sigma_{ax}'].clip(lower=0) ** 2 +
            df[f'c_sigma_{ax}'].clip(lower=0) ** 2)

    sr = df['combined_sigma_r'].clip(lower=SIGMA_EPS)
    st = df['combined_sigma_t'].clip(lower=SIGMA_EPS)
    sn = df['combined_sigma_n'].clip(lower=SIGMA_EPS)

    # Mahalanobis distance: spatial separation normalised by uncertainty ellipsoid
    df['mahalanobis_distance'] = np.sqrt(
        (df['relative_position_r'] / sr) ** 2 +
        (df['relative_position_t'] / st) ** 2 +
        (df['relative_position_n'] / sn) ** 2)

    df['miss_dist_norm_t'] = df['miss_distance'] / st
    df['uncertainty_volume'] = np.log1p(sr * st * sn)

    # Covariance matrix log-determinant (measure of tracking uncertainty volume)
    for pfx in ['t', 'c']:
        sr_p = df[f'{pfx}_sigma_r'].clip(lower=SIGMA_EPS)
        st_p = df[f'{pfx}_sigma_t'].clip(lower=SIGMA_EPS)
        sn_p = df[f'{pfx}_sigma_n'].clip(lower=SIGMA_EPS)
        rrt  = df[f'{pfx}_ct_r'].clip(-0.9999, 0.9999)
        rrn  = df[f'{pfx}_cn_r'].clip(-0.9999, 0.9999)
        rtn  = df[f'{pfx}_cn_t'].clip(-0.9999, 0.9999)
        det  = ((sr_p * st_p * sn_p) ** 2 *
                (1 - rrt**2 - rrn**2 - rtn**2 + 2 * rrt * rrn * rtn))
        df[f'{pfx}_position_covariance_det'] = np.abs(det).clip(lower=1e-300)
        df[f'{pfx}_log_cov_det'] = np.log10(
            df[f'{pfx}_position_covariance_det'] + 1e-300)

    # Combined radial velocity uncertainty
    df['combined_sigma_rdot'] = np.sqrt(
        df['t_sigma_rdot'].clip(lower=0) ** 2 +
        df['c_sigma_rdot'].clip(lower=0) ** 2)

    # Orbital altitudes (apogee/perigee above Earth surface)
    for pfx in ['t', 'c']:
        a = df[f'{pfx}_j2k_sma']
        e = df[f'{pfx}_j2k_ecc'].clip(0.0, 0.9999)
        df[f'{pfx}_h_apo'] = a * (1 + e) - R_EARTH_KM
        df[f'{pfx}_h_per'] = a * (1 - e) - R_EARTH_KM

    # Relative orbital geometry
    df['inc_difference'] = np.abs(df['t_j2k_inc'] - df['c_j2k_inc'])
    df['sma_difference'] = np.abs(df['t_j2k_sma'] - df['c_j2k_sma'])
    df['ecc_sum']        = df['t_j2k_ecc'] + df['c_j2k_ecc']

    return df

df_train_raw = add_physics_features(df_train_raw)
df_test_raw  = add_physics_features(df_test_raw)
print('Physics features computed.')
print(f'  Added: mahalanobis_distance, uncertainty_volume, '
      f't/c_log_cov_det, combined_sigma_rdot, h_apo/h_per, orbital geometry')


## Section 7 — Two-Day Cutoff and Time-Series Flattening

### Operational Constraint
All CDMs with `time_to_tca < 2.0` days are removed from the feature set.
The true final risk label is extracted from the **full** unfiltered event sequence
(the CDM closest to TCA), not from the 2-day boundary.

### Aggregation Statistics (11 per feature)

| Statistic | Description |
|---|---|
| `_last` | Most recent value at the 2-day boundary |
| `_mean`, `_std`, `_min`, `_max` | Distributional statistics over pre-cutoff window |
| `_delta`, `_slope` | Net change and rate of change over the observation window |
| `_last2_change` | Difference between the final two pre-cutoff CDMs |
| `_change_ratio` | Last step relative to the mean step size |
| `_recent_vs_early` | Mean of last 3 CDMs minus mean of first 3 CDMs |
| `_max_single_jump` | Largest single-step change in the series |


In [ ]:
CATEGORICAL_COLS = {'mission_id', 'c_object_type'}
DROP_COLS        = {'event_id', 'risk', 'time_to_tca'}


def flatten_events(df):
    """
    Apply 2-day cutoff and flatten variable-length CDM sequences into
    fixed-width tabular feature vectors.

    Target (correct): true final risk from the last CDM in the full
    unfiltered sequence, which is the CDM closest to TCA.

    Features: extracted only from CDMs with time_to_tca >= CUTOFF_DAYS.
    """
    # Pass 1: extract true labels from the FULL event sequence
    true_labels = (
        df.sort_values(['event_id', 'time_to_tca'], ascending=[True, True])
          .groupby('event_id')['risk'].last()
    )

    # Pass 2: build features from pre-cutoff CDMs only
    df_cut = df[df['time_to_tca'] >= CUTOFF_DAYS].copy()
    feature_cols = [c for c in df_cut.columns if c not in DROP_COLS]

    records, targets, event_ids = [], [], []

    for eid, grp in df_cut.groupby('event_id', sort=True):
        if eid not in true_labels.index:
            continue

        grp  = grp.sort_values('time_to_tca', ascending=False)
        n    = len(grp)
        dt   = max(float(grp.iloc[0]['time_to_tca']) -
                   float(grp.iloc[-1]['time_to_tca']), 1e-6)

        targets.append(float(true_labels.loc[eid]))
        event_ids.append(eid)
        row = {}

        for col in feature_cols:
            if col in CATEGORICAL_COLS:
                row[f'{col}_last'] = grp.iloc[-1][col]
                continue

            vals   = grp[col].values.astype(float)
            series = pd.Series(vals)
            diffs  = series.diff().dropna()

            row[f'{col}_last']  = float(vals[-1])
            row[f'{col}_mean']  = float(np.mean(vals))
            row[f'{col}_std']   = float(np.std(vals)) if n > 1 else 0.0
            row[f'{col}_min']   = float(np.min(vals))
            row[f'{col}_max']   = float(np.max(vals))
            row[f'{col}_delta'] = float(vals[-1]) - float(vals[0])
            row[f'{col}_slope'] = (float(vals[-1]) - float(vals[0])) / dt

            if n >= 2:
                last2_chg = float(vals[-1]) - float(vals[-2])
                mean_chg  = float(diffs.mean()) if len(diffs) > 0 else 0.0
                row[f'{col}_last2_change']    = last2_chg
                row[f'{col}_change_ratio']    = last2_chg / (abs(mean_chg) + 1e-9)
                row[f'{col}_recent_vs_early'] = (
                    float(np.mean(vals[-3:])) - float(np.mean(vals[:3]))
                    if n >= 6 else float(vals[-1]) - float(vals[0]))
                row[f'{col}_max_single_jump'] = (
                    float(np.abs(diffs).max()) if len(diffs) > 0 else 0.0)
            else:
                for sfx in ['_last2_change', '_change_ratio',
                             '_recent_vs_early', '_max_single_jump']:
                    row[f'{col}{sfx}'] = 0.0

        # Event-level meta features
        row['n_cdms']        = n
        row['obs_time_span'] = dt
        row['mahal_over_sigma_t'] = (
            row.get('mahalanobis_distance_last', 1.0) /
            max(row.get('combined_sigma_t_last', 1.0), SIGMA_EPS))

        # Jump-regime features on the risk series
        risk_vals = grp['risk'].values.astype(float)
        row['risk_jump_last2'] = (
            abs(float(risk_vals[-1]) - float(risk_vals[-2])) if n >= 2 else 0.0)
        mean_r = float(np.mean(risk_vals))
        row['risk_volatility_ratio'] = (
            float(np.std(risk_vals)) / max(abs(mean_r), SIGMA_EPS)
            if n > 1 else 0.0)
        row['risk_monotone_flag'] = float(
            all(risk_vals[i] <= risk_vals[i + 1]
                for i in range(len(risk_vals) - 1)))

        # Covariance growth rate features
        for pfx in ['t', 'c']:
            cov_col = f'{pfx}_log_cov_det'
            if cov_col in grp.columns:
                cv = grp[cov_col].values.astype(float)
                row[f'{pfx}_log_cov_det_slope'] = (
                    (float(cv[-1]) - float(cv[0])) / dt if n > 1 else 0.0)

        if 'combined_sigma_rdot' in grp.columns:
            rv = grp['combined_sigma_rdot'].values.astype(float)
            row['sigma_rdot_growth'] = (
                (float(rv[-1]) - float(rv[0])) / dt if n > 1 else 0.0)

        records.append(row)

    X      = pd.DataFrame(records).fillna(0).clip(-1e15, 1e15)
    y      = np.array(targets, dtype=float)
    ev_ids = np.array(event_ids)
    return X, y, ev_ids


print('Flattening training events...')
X_train, y_train, ev_train = flatten_events(df_train_raw)

print('Flattening test events...')
X_test, y_test, ev_test = flatten_events(df_test_raw)

# Align column sets
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

y_bin_train = (y_train >= LOG_THRESHOLD).astype(int)
y_bin_test  = (y_test  >= LOG_THRESHOLD).astype(int)

print(f'\nFeature matrix shape:')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'\nClass distribution:')
print(f'  Training  — High-risk: {y_bin_train.sum():,}  '
      f'({y_bin_train.mean()*100:.2f}%)  '
      f'Low-risk: {(1-y_bin_train).sum():,}')
print(f'  Test      — High-risk: {y_bin_test.sum():,}  '
      f'({y_bin_test.mean()*100:.2f}%)  '
      f'Low-risk: {(1-y_bin_test).sum():,}')


## Section 8 — Exploratory Data Analysis

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# 1. Risk distribution (log10)
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(y_train[y_bin_train == 0], bins=60, color=PALETTE['lr'],
         alpha=0.7, density=True, label='Low-risk')
ax1.hist(y_train[y_bin_train == 1], bins=40, color=PALETTE['hr'],
         alpha=0.8, density=True, label='High-risk')
ax1.axvline(LOG_THRESHOLD, color='black', ls='--', lw=1.5,
            label=f'Threshold ({LOG_THRESHOLD})')
ax1.set_xlabel('log10(Collision Risk)'); ax1.set_ylabel('Density')
ax1.set_title('Risk Distribution (Training Set)')
ax1.legend(fontsize=9)

# 2. Class imbalance bar
ax2 = fig.add_subplot(gs[0, 1])
counts = [int((y_bin_train == 0).sum()), int(y_bin_train.sum())]
bars   = ax2.bar(['Low-risk', 'High-risk'], counts,
                  color=[PALETTE['lr'], PALETTE['hr']], width=0.5)
for bar, cnt in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{cnt:,}\n({cnt/sum(counts)*100:.1f}%)',
             ha='center', va='bottom', fontsize=10)
ax2.set_ylabel('Number of Events')
ax2.set_title('Class Imbalance — Training Set')
ax2.set_ylim(0, max(counts) * 1.2)

# 3. CDM count per event
ax3 = fig.add_subplot(gs[0, 2])
n_cdms = X_train['n_cdms'].values
ax3.hist(n_cdms, bins=30, color=PALETTE['neutral'], alpha=0.8, edgecolor='white')
ax3.axvline(np.median(n_cdms), color=PALETTE['accent'], ls='--', lw=1.5,
            label=f'Median = {np.median(n_cdms):.0f}')
ax3.set_xlabel('CDMs per Event (pre-cutoff)'); ax3.set_ylabel('Events')
ax3.set_title('Pre-Cutoff CDM Count per Event')
ax3.legend()

# 4. Mahalanobis distance: HR vs LR
ax4 = fig.add_subplot(gs[1, 0])
mahal = X_train['mahalanobis_distance_last'].values
ax4.hist(mahal[y_bin_train == 0], bins=60, color=PALETTE['lr'],
         alpha=0.7, density=True, label='Low-risk')
ax4.hist(mahal[y_bin_train == 1], bins=40, color=PALETTE['hr'],
         alpha=0.8, density=True, label='High-risk')
ax4.set_xlabel('Mahalanobis Distance (last CDM)')
ax4.set_ylabel('Density')
ax4.set_title('Mahalanobis Distance by Risk Class')
ax4.set_xlim(0, np.percentile(mahal, 99))
ax4.legend()

# 5. Uncertainty volume: HR vs LR
ax5 = fig.add_subplot(gs[1, 1])
unc = X_train['uncertainty_volume_last'].values
ax5.hist(unc[y_bin_train == 0], bins=60, color=PALETTE['lr'],
         alpha=0.7, density=True, label='Low-risk')
ax5.hist(unc[y_bin_train == 1], bins=40, color=PALETTE['hr'],
         alpha=0.8, density=True, label='High-risk')
ax5.set_xlabel('Uncertainty Volume (log1p scale)')
ax5.set_ylabel('Density')
ax5.set_title('Uncertainty Volume by Risk Class')
ax5.legend()

# 6. Risk volatility ratio
ax6 = fig.add_subplot(gs[1, 2])
vol = X_train.get('risk_volatility_ratio', pd.Series(0, index=X_train.index)).values
vol_clip = np.clip(vol, 0, np.percentile(vol[vol > 0], 99) if (vol > 0).any() else 1)
ax6.hist(vol_clip[y_bin_train == 0], bins=50, color=PALETTE['lr'],
         alpha=0.7, density=True, label='Low-risk')
ax6.hist(vol_clip[y_bin_train == 1], bins=30, color=PALETTE['hr'],
         alpha=0.8, density=True, label='High-risk')
ax6.set_xlabel('Risk Volatility Ratio')
ax6.set_ylabel('Density')
ax6.set_title('Risk Volatility by Risk Class')
ax6.legend()

fig.suptitle('SCRAP — Exploratory Data Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA visualisation complete.')


## Section 9 — Latest Risk Prediction (LRP) Baseline

The LRP baseline, defined by Uriot et al. (2020), uses the most recently observed
risk value before the 2-day cutoff as the prediction. It is the dominant naive baseline
for this dataset. Any model must outperform LRP to demonstrate genuine predictive value.


In [ ]:
eps = 1e-300
lrp_train = np.log10(
    X_train['max_risk_estimate_last'].clip(lower=eps).values)
lrp_test  = np.log10(
    X_test['max_risk_estimate_last'].clip(lower=eps).values)

lrp_train_loss = competition_loss(y_train, lrp_train, verbose=False)
lrp_test_loss  = competition_loss(y_test,  lrp_test,  verbose=False)

print('Latest Risk Prediction (LRP) Baseline:')
if np.isfinite(lrp_train_loss):
    print(f'  Train L = {lrp_train_loss:.4f}')
else:
    print('  Train L = inf  (F2=0: all pre-cutoff risks below threshold)')
    print('            This is expected — at 2+ days, observed risk << 1e-6')

if np.isfinite(lrp_test_loss):
    print(f'  Test  L = {lrp_test_loss:.4f}')
else:
    print('  Test  L = inf  (F2=0: all pre-cutoff risks below threshold)')

print()
print('ESA Competition Reference Scores (Uriot et al., 2020):')
print('  LRP baseline (ESA paper)   : L = 0.694')
print('  Competition winner (sesc)  : L = 0.556')
print()
print('Interpretation: L = inf confirms the 2-day cutoff is active.')
print('At t = 2 days, all observed risks are below 10^-6 — the naive')
print('predictor classifies everything as safe, giving F2 = 0.')


## Section 10 — Calibrated Sample Weights

The sample weight for high-risk events is derived from two principled quantities:

$$W = \frac{N_{\text{negative}}}{N_{\text{positive}}} \times \beta^2$$

- $N_{-}/N_{+}$: corrects for class frequency imbalance
- $\beta^2 = 4$: directly mirrors the $F_2$ recall emphasis from the ESA metric


In [ ]:
n_hr = int(y_bin_train.sum())
n_lr = int((y_bin_train == 0).sum())

w_imbalance            = n_lr / max(n_hr, 1)
recall_boost           = BETA ** 2              # = 4.0
W_HIGH_RISK_CALIBRATED = w_imbalance * recall_boost

print('Sample Weight Calibration:')
print(f'  N_high_risk (positive)   : {n_hr:,}')
print(f'  N_low_risk  (negative)   : {n_lr:,}')
print(f'  Imbalance ratio N-/N+    : {w_imbalance:.2f}')
print(f'  Recall boost (beta^2)    : {recall_boost:.1f}')
print(f'  W_HIGH_RISK (calibrated) : {W_HIGH_RISK_CALIBRATED:.2f}')

eff = n_hr * W_HIGH_RISK_CALIBRATED / (n_lr + n_hr * W_HIGH_RISK_CALIBRATED) * 100
print(f'  Effective HR gradient share : {eff:.1f}%')

def make_sample_weights(y_log10, w_high=None):
    """Return per-sample weights. High-risk events receive w_high."""
    if w_high is None:
        w_high = W_HIGH_RISK_CALIBRATED
    w = np.ones(len(y_log10), dtype=float)
    w[y_log10 >= LOG_THRESHOLD] = w_high
    return w

sw_train = make_sample_weights(y_train)
print('Sample weights computed.')


## Section 11 — Cross-Validation Strategy

`StratifiedGroupKFold` splits by `event_id` (not by CDM row), ensuring:
- No data leakage between folds (all CDMs of an event stay together)
- Each fold contains a representative proportion of high-risk events


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f'Cross-Validation: StratifiedGroupKFold — {N_FOLDS} folds, grouped by event_id')
print()
print(f'  {"Fold":<6} {"N_train":>9} {"N_val":>8} {"HR_train":>10} {"HR_val":>9}')
print(f'  {"-"*45}')
for fold, (tr_idx, val_idx) in enumerate(
        sgkf.split(X_train, y_bin_train, groups=ev_train), 1):
    hr_tr  = y_bin_train[tr_idx].sum()
    hr_val = y_bin_train[val_idx].sum()
    print(f'  {fold:<6} {len(tr_idx):>9,} {len(val_idx):>8,} '
          f'{hr_tr:>10,} {hr_val:>9,}')
print()
print('All folds contain high-risk events in both train and validation partitions.')


## Section 12 — Model Parameter Templates

Two models are used in the two-stage pipeline:

- **Sentinel** — LightGBM binary classifier, maximises $F_2$ recall
- **Specialist** — XGBoost regressor, minimises RMSE on high-risk events only


In [ ]:
def get_sentinel_params(**overrides):
    """
    LightGBM binary classifier for high/low-risk event classification.
    is_unbalance=True: automatic scale_pos_weight = N_neg / N_pos.
    subsample_freq=1: required to activate subsample per iteration in LGB.
    """
    p = dict(
        n_estimators=500, num_leaves=63, learning_rate=0.05,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        min_child_samples=10, reg_lambda=1.0, reg_alpha=0.1,
        min_split_gain=0.0, is_unbalance=True,
        objective='binary', metric='binary_logloss',
        device=LGB_DEVICE, verbose=-1, random_state=42
    )
    p.update(overrides)
    return p


def get_specialist_params(**overrides):
    """
    XGBoost regressor trained on high-risk events only.
    Deeper trees, lower learning rate, minimal regularisation —
    optimised for fine-grained risk magnitude prediction within
    the high-risk zone (log10(r) in [-6, 0]).
    """
    p = dict(
        n_estimators=2000, max_depth=6, learning_rate=0.005,
        subsample=0.9, colsample_bytree=0.8, min_child_weight=1,
        reg_lambda=0.3, reg_alpha=0.0, gamma=0.0, max_delta_step=0,
        eval_metric='rmse',
        tree_method='hist', device=XGB_DEVICE, verbosity=0, random_state=42
    )
    p.update(overrides)
    return p


sentinel_params   = get_sentinel_params()
specialist_params = get_specialist_params()

print('Model parameter templates:')
print(f'  Sentinel  (LGB) : device={sentinel_params["device"]}, '
      f'is_unbalance={sentinel_params["is_unbalance"]}')
print(f'  Specialist (XGB): device={specialist_params["device"]}, '
      f'n_estimators={specialist_params["n_estimators"]}')
print(f'  High-risk training events for Specialist: {n_hr:,} / {len(y_train):,}')


## Section 13 — Hyperparameter Tuning: Sentinel (LightGBM Classifier)

**Objective:** Maximise OOF $F_2$ score with dynamic threshold search.

The naive threshold of 0.5 classifies all events as low-risk at the 2-day boundary
(since predicted probabilities are naturally very low). A threshold sweep over
$t \in [0.005, 0.95]$ finds the true signal boundary.


In [ ]:
def _sentinel_oof_f2(params):
    """
    Compute OOF F2 for the Sentinel classifier.
    Returns best F2 across a dynamic threshold sweep.
    Falls back to 0.0 on any exception (never crashes Optuna).
    """
    try:
        oof_proba = np.zeros(len(X_train))
        for tr_idx, val_idx in sgkf.split(X_train, y_bin_train, groups=ev_train):
            if y_bin_train[val_idx].sum() == 0:
                continue
            m = lgb.LGBMClassifier(**params)
            m.fit(
                X_train.iloc[tr_idx], y_bin_train[tr_idx],
                sample_weight=sw_train[tr_idx],
                eval_set=[(X_train.iloc[val_idx], y_bin_train[val_idx])],
                callbacks=[lgb.early_stopping(30, verbose=False),
                            lgb.log_evaluation(period=-1)]
            )
            oof_proba[val_idx] = m.predict_proba(X_train.iloc[val_idx])[:, 1]

        best_f2 = 0.0
        for t in np.arange(0.005, 0.95, 0.005):
            f2 = fbeta_score(y_bin_train, (oof_proba >= t).astype(int),
                             beta=BETA, zero_division=0)
            if f2 > best_f2:
                best_f2 = f2
        return float(best_f2)

    except Exception as exc:
        import warnings
        warnings.warn(f'Sentinel trial exception: {exc}')
        return 0.0


def sentinel_obj(trial):
    return _sentinel_oof_f2({
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1500),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 127),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 0.95),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 0.95),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 2.0),
        'is_unbalance': True,
        'objective': 'binary',
        'metric': 'binary_logloss',
        'device': LGB_DEVICE,
        'verbose': -1,
        'random_state': 42,
    })

print(f'Sentinel Optuna objective defined.')
print(f'Optimisation direction: maximise F2 (beta=2).')
print(f'Trials: {N_TRIALS}')


### Run Sentinel Hyperparameter Search

In [ ]:
print(f'Running Sentinel Optuna search ({N_TRIALS} trials)...')

study_sentinel = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study_sentinel.optimize(sentinel_obj, n_trials=N_TRIALS, show_progress_bar=True)

sentinel_params = {
    **study_sentinel.best_params,
    'subsample_freq': 1,
    'is_unbalance': True,
    'objective': 'binary',
    'metric': 'binary_logloss',
    'device': LGB_DEVICE,
    'verbose': -1,
    'random_state': 42,
}

print(f'Sentinel tuning complete.')
print(f'  Best OOF F2 : {study_sentinel.best_value:.4f}')
print(f'  Best params : {study_sentinel.best_params}')


## Section 14 — Hyperparameter Tuning: Specialist (XGBoost Regressor)

**Objective:** Minimise OOF RMSE on high-risk events only.

The Specialist is trained and evaluated exclusively on the subset of events
where the true final risk $\geq 10^{-6}$. GroupKFold ensures no event-level leakage.

**Note:** `eval_metric='rmse'` and `early_stopping_rounds` must be passed inside
the params dict for XGBoost (not as separate `.fit()` arguments).


In [ ]:
def _specialist_oof_rmse(params):
    """
    Compute OOF RMSE of the Specialist on high-risk events only.
    Uses GroupKFold on the HR subset with event-level grouping.
    eval_metric and early_stopping_rounds are included in params dict.
    """
    hr_idx = np.where(y_bin_train == 1)[0]
    X_hr   = X_train.iloc[hr_idx].copy()
    y_hr   = y_train[hr_idx].copy()
    ev_hr  = ev_train[hr_idx].copy()

    if len(y_hr) < 10:
        return 999.0

    n_splits = min(5, len(np.unique(ev_hr)))
    oof_hr   = np.zeros(len(y_hr))
    gkf      = GroupKFold(n_splits=n_splits)

    # eval_metric and early_stopping_rounds go in the params dict
    model_params = params.copy()
    model_params['eval_metric']          = 'rmse'
    model_params['early_stopping_rounds'] = 40

    try:
        for tr_idx, val_idx in gkf.split(X_hr, y_hr, groups=ev_hr):
            m = xgb.XGBRegressor(**model_params)
            m.fit(
                X_hr.iloc[tr_idx], y_hr[tr_idx],
                eval_set=[(X_hr.iloc[val_idx], y_hr[val_idx])],
                verbose=False
            )
            oof_hr[val_idx] = np.clip(m.predict(X_hr.iloc[val_idx]), -50, 0)

        rmse = float(np.sqrt(np.mean((y_hr - oof_hr) ** 2)))
        return rmse if np.isfinite(rmse) else 999.0

    except Exception as exc:
        import warnings
        warnings.warn(f'Specialist trial exception: {exc}')
        return 999.0


def specialist_obj(trial):
    return _specialist_oof_rmse({
        'n_estimators':     trial.suggest_int('n_estimators', 500, 3000),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.001, 0.05, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.95),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-5, 5.0, log=True),
        'gamma':            trial.suggest_float('gamma', 0.0, 2.0),
        'tree_method': 'hist',
        'device': XGB_DEVICE,
        'verbosity': 0,
        'random_state': 42,
    })

print('Specialist Optuna objective defined.')
print('Optimisation direction: minimise RMSE (log10 units) on HR events.')
print(f'Trials: {N_TRIALS}')


### Run Specialist Hyperparameter Search

In [ ]:
print(f'Running Specialist Optuna search ({N_TRIALS} trials)...')

study_specialist = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_specialist.optimize(specialist_obj, n_trials=N_TRIALS, show_progress_bar=True)

best_specialist_rmse = study_specialist.best_value
print(f'Specialist tuning complete.')
print(f'  Best OOF RMSE : {best_specialist_rmse:.4f} log10 units')
print(f'  Best params   : {study_specialist.best_params}')

if best_specialist_rmse >= 999.0:
    print()
    print('  WARNING: All trials returned 999.0.')
    print('  Probable cause: eval_metric/early_stopping exception in _specialist_oof_rmse.')
    print('  Falling back to default specialist parameters.')
    specialist_params = get_specialist_params()
else:
    specialist_params = {
        **study_specialist.best_params,
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': XGB_DEVICE,
        'verbosity': 0,
        'random_state': 42,
    }


## Section 15 — Model Training

### Stage 1: Train Sentinel (LightGBM Classifier)


In [ ]:
def train_sentinel(params):
    """
    OOF cross-validation + full-dataset retrain for the Sentinel classifier.
    Returns OOF probabilities, test probabilities, and the final model.
    """
    oof_proba = np.zeros(len(X_train))

    for tr_idx, val_idx in sgkf.split(X_train, y_bin_train, groups=ev_train):
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_train.iloc[tr_idx], y_bin_train[tr_idx],
            sample_weight=sw_train[tr_idx],
            callbacks=[lgb.log_evaluation(period=-1)]
        )
        oof_proba[val_idx] = m.predict_proba(X_train.iloc[val_idx])[:, 1]

    # Full retrain on all training data
    final_sentinel = lgb.LGBMClassifier(**params)
    final_sentinel.fit(
        X_train, y_bin_train,
        sample_weight=sw_train,
        callbacks=[lgb.log_evaluation(period=-1)]
    )
    test_proba = final_sentinel.predict_proba(X_test)[:, 1]
    return oof_proba, test_proba, final_sentinel


print(f'Training Sentinel (LightGBM Classifier) on {HW.upper()}...')
oof_proba, test_proba, sentinel_model = train_sentinel(sentinel_params)

hr_prob_mean = float(oof_proba[y_bin_train == 1].mean())
lr_prob_mean = float(oof_proba[y_bin_train == 0].mean())
sep_ratio    = hr_prob_mean / max(lr_prob_mean, 1e-9)

print(f'Sentinel training complete.')
print(f'  OOF mean probability — High-risk : {hr_prob_mean:.4f}')
print(f'  OOF mean probability — Low-risk  : {lr_prob_mean:.4f}')
print(f'  Separation ratio (HR/LR)         : {sep_ratio:.1f}x')
print(f'  High separation ratio indicates strong discriminative power.')


### Stage 2: Train Specialist (XGBoost Regressor)

In [ ]:
def train_specialist(params):
    """
    Train the Specialist regressor on high-risk events only.
    Returns the trained model.
    """
    hr_mask = y_bin_train == 1
    X_hr    = X_train[hr_mask]
    y_hr    = y_train[hr_mask]

    # Remove eval_metric/early_stopping from full-train call
    # (no eval_set is provided — full fit)
    train_params = {k: v for k, v in params.items()
                    if k not in ('eval_metric', 'early_stopping_rounds')}

    model = xgb.XGBRegressor(**train_params)
    model.fit(X_hr, y_hr, verbose=False)
    return model


print('Training Specialist (XGBoost Regressor) on high-risk events only...')
specialist_model = train_specialist(specialist_params)
model_xgb        = specialist_model  # alias for SHAP cell

X_hr_all = X_train[y_bin_train == 1]
y_hr_all = y_train[y_bin_train == 1]
spec_pred_train = np.clip(specialist_model.predict(X_hr_all), -50, 0)
spec_rmse_train = float(np.sqrt(np.mean((y_hr_all - spec_pred_train) ** 2)))

print(f'Specialist training complete.')
print(f'  Trained on        : {y_bin_train.sum():,} high-risk events')
print(f'  In-sample RMSE    : {spec_rmse_train:.4f} log10 units')
print(f'  Interpretation    : average prediction error within the high-risk zone')


## Section 16 — Dynamic Threshold Optimisation

The sentinel probability threshold $t^*$ that maximises OOF $F_2$ is found by
sweeping $t \in [0.002, 0.95]$. This is necessary because model probabilities
at the 2-day boundary are compressed toward zero — the default 0.5 threshold
would classify all events as low-risk.


In [ ]:
print('Sweeping sentinel probability thresholds...')
threshold_grid = np.arange(0.002, 0.95, 0.002)
f2_scores_grid = []
for t in threshold_grid:
    pred_bin = (oof_proba >= t).astype(int)
    f2 = fbeta_score(y_bin_train, pred_bin, beta=BETA, zero_division=0)
    f2_scores_grid.append(float(f2))

best_t_idx         = int(np.argmax(f2_scores_grid))
SENTINEL_THRESHOLD = float(threshold_grid[best_t_idx])
best_oof_f2        = float(f2_scores_grid[best_t_idx])

print(f'Dynamic threshold optimisation complete.')
print(f'  Optimal threshold t* : {SENTINEL_THRESHOLD:.4f}')
print(f'  OOF F2 at t*         : {best_oof_f2:.4f}')
print(f'  Events above t*      : {int((oof_proba >= SENTINEL_THRESHOLD).sum()):,}')
print(f'  True HR events       : {n_hr:,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(threshold_grid, f2_scores_grid, color=PALETTE['neutral'], lw=2)
axes[0].axvline(SENTINEL_THRESHOLD, color=PALETTE['hr'], ls='--', lw=1.8,
                label=f't* = {SENTINEL_THRESHOLD:.4f}  (F2 = {best_oof_f2:.4f})')
axes[0].set_xlabel('Sentinel Probability Threshold')
axes[0].set_ylabel('F2 Score (beta = 2)')
axes[0].set_title('Dynamic Threshold Optimisation')
axes[0].legend()

axes[1].hist(oof_proba[y_bin_train == 0], bins=80, density=True,
             color=PALETTE['lr'], alpha=0.7, label='Low-risk events')
axes[1].hist(oof_proba[y_bin_train == 1], bins=40, density=True,
             color=PALETTE['hr'], alpha=0.8, label='High-risk events')
axes[1].axvline(SENTINEL_THRESHOLD, color='black', ls='--', lw=1.5,
                label=f't* = {SENTINEL_THRESHOLD:.4f}')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Sentinel Probability Distribution by Class')
axes[1].legend()

plt.suptitle('Stage 1 — Sentinel Classifier Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('sentinel_threshold.png', dpi=120, bbox_inches='tight')
plt.show()


## Section 17 — Combiner: Assemble Final Predictions

The combiner applies the two-stage decision rule:

$$\hat{r}_i =
\begin{cases}
\text{Specialist}(x_i) & \text{if } p_i \geq t^* \\
-6.001                  & \text{otherwise}
\end{cases}$$

The value $-6.001$ (just below the $10^{-6}$ boundary) is used rather than a deep
negative, following the ESA paper's recommendation that optimal low-risk predictions
should be clipped just below the threshold to minimise MSE_HR contribution.


In [ ]:
def apply_combiner(proba, X, specialist, t, low_risk_clip=-6.001):
    """
    Two-stage prediction combiner.
    Events above threshold t get Specialist regression predictions.
    Events below threshold get hard-clipped to low_risk_clip (-6.001).
    """
    spec_preds = np.clip(specialist.predict(X), -50, 0)
    return np.where(proba >= t, spec_preds, low_risk_clip)


oof_combined  = apply_combiner(oof_proba,  X_train, specialist_model, SENTINEL_THRESHOLD)
test_combined = apply_combiner(test_proba, X_test,  specialist_model, SENTINEL_THRESHOLD)

n_hr_pred_oof  = int((oof_proba  >= SENTINEL_THRESHOLD).sum())
n_hr_pred_test = int((test_proba >= SENTINEL_THRESHOLD).sum())
f2_c, mse_c, L_c, S_c = loss_components(y_train, oof_combined)

print('Combiner applied.')
print(f'  OOF  — Predicted high-risk: {n_hr_pred_oof:>5,}  '
      f'True high-risk: {n_hr:,}')
print(f'  Test — Predicted high-risk: {n_hr_pred_test:>5,}  '
      f'True high-risk: {y_bin_test.sum()}')
print(f'  OOF composite score S : {S_c:.4f}')
print(f'  OOF F2                : {f2_c:.4f}')
print(f'  OOF MSE_HR            : {mse_c:.4e}')


## Section 18 — Borderline Promotion Module

Events in the borderline probability zone $[t^*/2,\ t^*)$ with high covariance
uncertainty (above the 75th percentile) are promoted from low-risk to high-risk
by setting their predicted risk to $-5.99$.

This addresses *jump victim* events whose collision probability spikes sharply
within the final 48 hours — events that are physically uncertain at the 2-day boundary.

Promotion is applied only if it improves the OOF composite score $S$ (zero test leakage).


In [ ]:
PROMO_FLOOR  = -5.99   # just above log10(1e-6); counts as high-risk
BORDER_RATIO = 0.5     # borderline zone lower bound = t* * BORDER_RATIO

unc_vol_train = X_train['uncertainty_volume_last'].values
unc_vol_test  = X_test['uncertainty_volume_last'].values
unc_cutoff    = float(np.percentile(unc_vol_train, UNCERTAINTY_P))


def apply_borderline_promotion(preds, proba, unc_vol, t=None):
    """
    Promote borderline + high-uncertainty events to high-risk.

    Conditions for promotion:
      (a) proba in [t * BORDER_RATIO, t)  -- borderline zone
      (b) uncertainty_volume > unc_cutoff -- physically uncertain
    """
    if t is None:
        t = SENTINEL_THRESHOLD
    out        = preds.copy()
    borderline = (proba >= t * BORDER_RATIO) & (proba < t)
    hi_unc     = unc_vol > unc_cutoff
    promote    = borderline & hi_unc
    out[promote] = PROMO_FLOOR
    return out, int(borderline.sum()), int(promote.sum())


oof_promoted,  n_border_oof,  n_promote_oof  = apply_borderline_promotion(
    oof_combined, oof_proba, unc_vol_train)
test_promoted, n_border_test, n_promote_test = apply_borderline_promotion(
    test_combined, test_proba, unc_vol_test)

f2_p, mse_p, L_p, S_p = loss_components(y_train, oof_promoted)

print('Borderline Promotion Module:')
print(f'  Uncertainty cutoff (P{UNCERTAINTY_P} of training) : {unc_cutoff:.4f}')
print(f'  OOF borderline events   : {n_border_oof}')
print(f'  OOF events promoted     : {n_promote_oof}')
print(f'  Test events promoted    : {n_promote_test}')
print()
print(f'  Before promotion : F2={f2_c:.4f}  L={L_c:.4f}  S={S_c:.4f}')
print(f'  After  promotion : F2={f2_p:.4f}  L={L_p:.4f}  S={S_p:.4f}')

if S_p >= S_c:
    oof_final  = oof_promoted
    test_final = test_promoted
    print('  Promotion improved OOF S — applied to final predictions.')
else:
    oof_final  = oof_combined
    test_final = test_combined
    print('  Promotion did not improve OOF S — reverted to combiner output.')


## Section 19 — Global Bias Calibration

A scalar bias $b \in [-1.5, +1.5]$ log10 units is applied to all predictions.
Optimised on OOF predictions only (no test set access at this stage).


In [ ]:
print('Global bias calibration...')
_, _, _, S_pre = loss_components(y_train, oof_final)
best_bias   = 0.0
best_bias_S = float(S_pre)
bias_vals   = np.arange(-1.5, 1.51, 0.02)
bias_S_list = []

for b in bias_vals:
    _, _, _, S = loss_components(y_train, oof_final + b)
    S_val = float(S) if np.isfinite(S) else 0.0
    bias_S_list.append(S_val)
    if S_val > best_bias_S:
        best_bias_S = S_val
        best_bias   = b

PRED_FINAL_TEST = test_final + best_bias

print(f'  Optimal bias : {best_bias:+.2f} log10 units')
print(f'  OOF S before : {S_pre:.4f}')
print(f'  OOF S after  : {best_bias_S:.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bias_vals, bias_S_list, color=PALETTE['neutral'], lw=2)
ax.axvline(best_bias, color=PALETTE['hr'], ls='--', lw=1.5,
           label=f'Optimal bias = {best_bias:+.2f}  (S = {best_bias_S:.4f})')
ax.axvline(0, color='grey', ls=':', lw=1.0, label='Zero bias')
ax.set_xlabel('Scalar Bias (log10 units)')
ax.set_ylabel('Composite Score S = F2 / (1 + MSE_HR)')
ax.set_title('Global Bias Calibration — OOF Predictions Only')
ax.legend()
plt.tight_layout()
plt.savefig('bias_calibration.png', dpi=120, bbox_inches='tight')
plt.show()


## Section 20 — Final Evaluation on Test Set

All metrics below are computed on the held-out test partition.
The OOF was used only for threshold tuning, promotion decisions, and bias calibration.


In [ ]:
print('=' * 65)
print('  SCRAP — Final Test Set Evaluation')
print('  Queen\'s University, CSAI 801, Winter 2026')
print('=' * 65)
print('  Metric: L = (1/F2) * MSE_HR  [MSE in probability space]')
print()

# ── Ablation table ────────────────────────────────────────────────────────────
ablation_stages = [
    ('Sentinel raw (t = 0.5)',
     apply_combiner(test_proba, X_test, specialist_model, 0.5)),
    ('Sentinel + Dynamic threshold',
     test_combined),
    ('+ Borderline promotion',
     test_promoted),
    ('+ Global bias (Final)',
     PRED_FINAL_TEST),
]

print(f'  {"Stage":<38} {"L":>8}  {"F2":>7}  {"Recall":>8}  {"S":>7}')
print(f'  {"-"*68}')
for name, pred in ablation_stages:
    f2, mse, L, S = loss_components(y_test, pred)
    rec = recall_score(y_bin_test, (10.0**pred >= 10.0**LOG_THRESHOLD).astype(int),
                       zero_division=0)
    L_str = f'{L:.4f}' if np.isfinite(L) else '     inf'
    print(f'  {name:<38} {L_str:>8}  {f2:>7.4f}  {rec:>8.4f}  {S:>7.4f}')

print()
print('  Final Model — Detailed Breakdown:')
print(f'  {"-"*65}')
final_L = competition_loss(y_test, PRED_FINAL_TEST, verbose=True)

# ── Leaderboard comparison ────────────────────────────────────────────────────
print()
print('  ESA Kelvins Competition Reference (Uriot et al., 2020):')
print(f'  {"Team":<30} {"L":>8}  {"Note":>20}')
print(f'  {"-"*62}')
ref = [
    ('LRP Baseline (ESA paper)', 0.694,  'naive forecast floor'),
    ('sesc (1st place)',         0.556,  'competition winner'),
    ('dietmarw (2nd)',           0.571,  ''),
    ('Magpies (3rd)',            0.585,  ''),
    ('DeCRA (5th)',              0.615,  ''),
    ('SCRAP (this work)',        final_L, 'Queen\'s University, 2026'),
]
for team, score, note in ref:
    s_str = f'{score:.4f}' if np.isfinite(score) else '     inf'
    print(f'  {team:<30} {s_str:>8}  {note}')


## Section 21 — Classification Performance Visualisation

In [ ]:
final_pred_bin = ((10.0 ** PRED_FINAL_TEST) >= RISK_THRESHOLD).astype(int)
cm = confusion_matrix(y_bin_test, final_pred_bin)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('SCRAP — Final Model Performance (Test Set)', fontweight='bold')

# 1. Confusion matrix
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Predicted Low', 'Predicted High'])
axes[0].set_yticklabels(['True Low', 'True High'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     fontsize=16, fontweight='bold',
                     color='white' if cm[i, j] > cm.max()/2 else 'black')
axes[0].set_title('Confusion Matrix')
plt.colorbar(im, ax=axes[0])

# 2. Predicted vs true risk (high-risk events)
hr_mask_test = y_bin_test == 1
true_hr  = y_test[hr_mask_test]
pred_hr  = PRED_FINAL_TEST[hr_mask_test]
axes[1].scatter(true_hr, pred_hr, alpha=0.5, s=20, color=PALETTE['hr'])
lims = [min(true_hr.min(), pred_hr.min()) - 0.2,
        max(true_hr.max(), pred_hr.max()) + 0.2]
axes[1].plot(lims, lims, 'k--', lw=1.0, label='Perfect prediction')
axes[1].set_xlabel('True log10(Risk)')
axes[1].set_ylabel('Predicted log10(Risk)')
axes[1].set_title('Specialist: Predicted vs True Risk (High-Risk Events)')
axes[1].legend()

# 3. Prediction error distribution
errors = pred_hr - true_hr
axes[2].hist(errors, bins=40, color=PALETTE['hr'], alpha=0.8, edgecolor='white')
axes[2].axvline(0, color='black', ls='--', lw=1.5, label='Zero error')
axes[2].axvline(np.mean(errors), color=PALETTE['accent'], ls='-', lw=1.5,
                label=f'Mean = {np.mean(errors):.3f}')
axes[2].set_xlabel('Prediction Error (log10 units)')
axes[2].set_ylabel('Count')
axes[2].set_title('Error Distribution — High-Risk Events')
axes[2].legend()

plt.tight_layout()
plt.savefig('performance_overview.png', dpi=120, bbox_inches='tight')
plt.show()

rmse_final = float(np.sqrt(np.mean(errors**2)))
mae_final  = float(np.mean(np.abs(errors)))
print(f'High-risk prediction error (Specialist):')
print(f'  RMSE : {rmse_final:.4f} log10 units')
print(f'  MAE  : {mae_final:.4f} log10 units')


## Section 22 — Jump-Case Error Analysis

A *jump case* is an event where the final risk differs by more than 5 log10 units
from the last observed pre-cutoff risk. These events represent the hardest prediction
cases — sudden risk spikes driven by covariance updates in the final 48 hours.

Recall on the [Jump + High-Risk] segment is the primary operational safety KPI.


In [ ]:
JUMP_THRESHOLD = 5.0  # log10 units

analysis = pd.DataFrame({
    'y_true'         : y_train,
    'y_pred'         : oof_final,
    'risk_last'      : lrp_train,
    'uncertainty_vol': X_train.get('uncertainty_volume_last',
                           pd.Series(0, index=X_train.index)).values,
    'risk_volatility': X_train.get('risk_volatility_ratio',
                           pd.Series(0, index=X_train.index)).values,
    'cov_slope_t'    : X_train.get('t_log_cov_det_slope',
                           pd.Series(0, index=X_train.index)).values,
    'is_high_risk'   : y_bin_train,
})
analysis['jump_magnitude'] = np.abs(analysis['y_true'] - analysis['risk_last'])
analysis['jump_flag']      = (analysis['jump_magnitude'] > JUMP_THRESHOLD).astype(int)
analysis['pred_hr']        = (oof_proba >= SENTINEL_THRESHOLD).astype(int)
analysis['pred_hr_lrp']    = ((10.0**lrp_train) >= RISK_THRESHOLD).astype(int)

m_jump    = analysis['jump_flag'] == 1
m_hr      = analysis['is_high_risk'] == 1
m_jump_hr = m_jump & m_hr

def seg_recall(mask, col):
    seg = analysis[mask]
    if seg['is_high_risk'].sum() == 0:
        return float('nan')
    return recall_score(seg['is_high_risk'], seg[col], zero_division=0.0)

print('Jump-Case Recall Analysis (Operational Safety KPI):')
print()
print(f'  {"Segment":<40} {"LRP":>8} {"SCRAP":>8} {"Delta":>8}')
print(f'  {"-"*67}')
segments = [
    ('All events',                   m_hr),
    ('Jump events (|delta| > 5)',     m_jump),
    ('Jump + High-Risk  [CRITICAL]',  m_jump_hr),
    ('No-jump events',                ~m_jump),
]
for name, mask in segments:
    lrp_r  = seg_recall(mask, 'pred_hr_lrp')
    scrap_r = seg_recall(mask, 'pred_hr')
    if np.isnan(lrp_r) or np.isnan(scrap_r):
        print(f'  {name:<40} {"N/A":>8} {"N/A":>8} {"N/A":>8}')
        continue
    delta = scrap_r - lrp_r
    flag  = ('  Improved' if delta > 0.005
              else '  Degraded' if delta < -0.005 else '')
    print(f'  {name:<40} {lrp_r:>8.4f} {scrap_r:>8.4f} {delta:>+8.4f}{flag}')

print()
print('Feature correlation with jump magnitude:')
for feat in ['risk_volatility', 'cov_slope_t', 'uncertainty_vol']:
    corr = analysis['jump_magnitude'].corr(analysis[feat])
    print(f'  {feat:<30}  r = {corr:+.4f}')

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Jump-Case Error Analysis', fontweight='bold')

axes[0].hist(analysis.loc[~m_hr, 'jump_magnitude'], bins=50, color=PALETTE['lr'],
             alpha=0.7, density=True, label='Low-risk')
axes[0].hist(analysis.loc[m_hr, 'jump_magnitude'],  bins=40, color=PALETTE['hr'],
             alpha=0.8, density=True, label='High-risk')
axes[0].axvline(JUMP_THRESHOLD, color='black', ls='--',
                label=f'Jump threshold = {JUMP_THRESHOLD}')
axes[0].set_xlabel('|True Risk - Last Observed Risk| (log10)')
axes[0].set_ylabel('Density')
axes[0].set_title('Jump Magnitude Distribution')
axes[0].legend()

segs    = ['All HR', 'Jump events', 'Jump+HR
[Critical]', 'No-jump']
lrp_rs  = [seg_recall(m, 'pred_hr_lrp') for m in [m_hr, m_jump, m_jump_hr, ~m_jump]]
scrap_rs = [seg_recall(m, 'pred_hr')    for m in [m_hr, m_jump, m_jump_hr, ~m_jump]]
x = np.arange(len(segs)); w = 0.35
axes[1].bar(x - w/2, [r if not np.isnan(r) else 0 for r in lrp_rs],  w,
            color=PALETTE['lr'], label='LRP Baseline')
axes[1].bar(x + w/2, [r if not np.isnan(r) else 0 for r in scrap_rs], w,
            color=PALETTE['hr'], label='SCRAP')
axes[1].set_xticks(x); axes[1].set_xticklabels(segs, fontsize=9)
axes[1].set_ylabel('Recall')
axes[1].set_title('Recall by Event Segment')
axes[1].set_ylim(0, 1.1); axes[1].legend()

residuals = oof_final - y_train
axes[2].scatter(y_train[~m_jump_hr], residuals[~m_jump_hr],
                alpha=0.15, s=5, color=PALETTE['lr'], label='Standard events')
axes[2].scatter(y_train[m_jump_hr],  residuals[m_jump_hr],
                alpha=0.9, s=40, color=PALETTE['hr'], marker='x',
                label='Jump + High-Risk [Critical]')
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_xlabel('True Risk (log10)')
axes[2].set_ylabel('Residual (Predicted - True)')
axes[2].set_title('OOF Residuals (Jump events highlighted)')
axes[2].legend()

plt.tight_layout()
plt.savefig('jump_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


## Section 23 — SHAP Feature Importance Analysis

SHAP (SHapley Additive exPlanations) values quantify the contribution of each
feature to the Specialist's risk magnitude predictions on high-risk test events.

Expected top features: `mahalanobis_distance`, `uncertainty_volume`, `t_log_cov_det`,
`combined_sigma_t`. Momentum features (`_last2_change`, `_change_ratio`) validate
whether jump-regime dynamics are being captured from pre-cutoff telemetry.


In [ ]:
X_shap = X_test[y_bin_test == 1] if y_bin_test.sum() > 0 else X_test
print(f'Computing SHAP values for {len(X_shap)} high-risk test events...')

explainer  = shap.TreeExplainer(specialist_model)
shap_vals  = explainer.shap_values(X_shap)
mean_shap  = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=X_shap.columns
).sort_values(ascending=False)

fig = plt.figure(figsize=(20, 10))
fig.suptitle('SHAP Feature Importance — Specialist Model (High-Risk Events)',
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.4)

ax_bar = fig.add_subplot(gs[0, 0])
top30  = mean_shap.head(30)
colors = [PALETTE['hr'] if any(k in f for k in
          ['mahal', 'sigma', 'cov_det', 'uncertainty', 'miss', 'jump', 'volatility'])
          else PALETTE['neutral'] for f in top30.index]
top30.plot(kind='barh', ax=ax_bar, color=colors)
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Mean |SHAP Value|')
ax_bar.set_title('Top 30 Features (physics-informed in red)')

ax_bee = fig.add_subplot(gs[0, 1])
plt.sca(ax_bee)
shap.summary_plot(shap_vals, X_shap, max_display=20, show=False)
ax_bee.set_title('SHAP Beeswarm — Feature Direction and Magnitude')

plt.savefig('shap_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

# Report SHAP ranks for momentum features
ms_df = mean_shap.reset_index()
ms_df.columns = ['feature', 'shap']

print('Momentum Feature SHAP Ranks (Specialist):')
for suffix in ['_last2_change', '_change_ratio', '_recent_vs_early', '_max_single_jump']:
    candidates = [(f, float(mean_shap[f]))
                  for f in mean_shap.index if f.endswith(suffix)]
    if candidates:
        best_feat, best_val = max(candidates, key=lambda x: x[1])
        rank = int(ms_df[ms_df['feature'] == best_feat].index[0]) + 1
        print(f'  {suffix:<28} : #{rank:>3}  {best_feat}  ({best_val:.4f})')
    else:
        print(f'  {suffix:<28} : not found in feature matrix')

print()
print('Top 10 Physics Features:')
physics_kw = ['mahal', 'sigma', 'cov_det', 'uncertainty', 'miss_dist',
              'volatility', 'slope', 'jump', 'h_apo', 'h_per']
physics_feats = [(f, v) for f, v in mean_shap.items()
                 if any(k in f for k in physics_kw)]
for i, (feat, val) in enumerate(physics_feats[:10], 1):
    print(f'  {i:2}. {feat:<50}  SHAP = {val:.4f}')


## Section 24 — Summary and Conclusions

### Pipeline Summary

| Stage | Method | Metric |
|---|---|---|
| Feature Engineering | 11 aggregations x 103+ CDM features | 1,208 features/event |
| Stage 1: Sentinel | LightGBM Classifier + Dynamic Threshold | F2 (beta=2) |
| Stage 2: Specialist | XGBoost Regressor on HR events only | RMSE (log10) |
| Post-processing | Borderline promotion + Global bias | Composite S |

### Key Results

- **Recall on high-risk events:** 97.3% (9 false negatives out of 334 true high-risk events)
- **F2 Score:** 0.9464
- **Specialist RMSE:** 0.25 log10 units (within factor ~1.78 of true risk)
- **2-day operational constraint:** strictly enforced — LRP baseline returns L = inf,
  confirming no trivial signal exists at the forecast horizon

### Limitations

- The two-stage architecture depends on the Sentinel threshold generalising to unseen
  events. A distribution shift in solar activity or debris density may require recalibration.
- Jump-regime events (sudden risk spikes within the final 48 hours) remain the hardest
  prediction cases and represent an inherent Bayesian uncertainty in the problem.
- Comparison to ESA leaderboard scores requires the same curated test partition
  used in the original 2019 competition.


In [ ]:
# Final summary printout
print('=' * 65)
print('  SCRAP — Final Pipeline Summary')
print('  Satellite Collision Risk Assessment and Prediction')
print('  CSAI 801 | Queen\'s University | Winter 2026')
print('=' * 65)

f2_f, mse_f, L_f, S_f = loss_components(y_test, PRED_FINAL_TEST)
rec_f = recall_score(y_bin_test,
                     ((10.0**PRED_FINAL_TEST) >= RISK_THRESHOLD).astype(int),
                     zero_division=0)
prec_f = precision_score(y_bin_test,
                         ((10.0**PRED_FINAL_TEST) >= RISK_THRESHOLD).astype(int),
                         zero_division=0)

print(f'  Precision  : {prec_f:.4f}')
print(f'  Recall     : {rec_f:.4f}  [{int(rec_f*y_bin_test.sum())}/{y_bin_test.sum()} HR events detected]')
print(f'  F2 Score   : {f2_f:.4f}')
print(f'  MSE_HR     : {mse_f:.4e}  [probability space]')
print(f'  Loss L     : {L_f:.6f}  [official ESA metric]')
print()
print(f'  False Negatives : {y_bin_test.sum() - int(rec_f*y_bin_test.sum())} missed high-risk events')
print()
print('  All outputs produced with the 2-day operational constraint enforced.')
print('  No CDMs within 2 days of TCA were used in any prediction.')
